In [1]:
from pathlib import Path

import pandas as pd

# Build Pair-Divergence Universe

Construct a reproducible US stock universe from current Nasdaq listing metadata for downstream Yahoo Finance data collection.

Security type is inferred from the available metadata, and the resulting current-snapshot universe is subject to survivorship bias.

In [2]:
currentDirectory = Path.cwd()

possibleRoots = [
    currentDirectory,
    currentDirectory.parent,
    currentDirectory.parent.parent
]

repositoryRoot = None

for directory in possibleRoots:
    universeDirectory = directory /"data"/"universes"

    if universeDirectory.exists():
        repositoryRoot = directory
        break

if repositoryRoot is None:
    raise FileNotFoundError("Could not find data/universes")

universeDirectory = repositoryRoot/"data"/"universes"

## Load Source Data

In [3]:
# Define source file paths
nasdaqPath = universeDirectory/"nasdaqlisted.txt"
otherPath = universeDirectory/"otherlisted.txt"

# Verify required files exist
if not nasdaqPath.exists():
    raise FileNotFoundError(f"Missing source file: {nasdaqPath}")

if not otherPath.exists():
    raise FileNotFoundError(f"Missing source file: {otherPath}")

# Load all source data as strings
nasdaqListings = pd.read_csv(
    nasdaqPath,
    sep="|",
    dtype=str,
    keep_default_na=False
)

otherListings = pd.read_csv(
    otherPath,
    sep="|",
    dtype=str,
    keep_default_na=False
)

# Validate required source columns
requiredNasdaqColumns = {
    "Symbol",
    "Security Name",
    "ETF",
    "Test Issue",
    "Financial Status"
}

requiredOtherColumns = {
    "ACT Symbol",
    "NASDAQ Symbol",
    "Security Name",
    "Exchange",
    "ETF",
    "Test Issue"
}

missingNasdaqColumns = requiredNasdaqColumns - set(nasdaqListings.columns)
missingOtherColumns = requiredOtherColumns - set(otherListings.columns)

if missingNasdaqColumns:
    raise ValueError(f"Missing Nasdaq columns: {sorted(missingNasdaqColumns)}")

if missingOtherColumns:
    raise ValueError(f"Missing other-listing columns: {sorted(missingOtherColumns)}")

In [4]:
# Remove footer rows from exchange directories
nasdaqListings = nasdaqListings[~nasdaqListings["Symbol"].str.startswith("File Creation Time")].reset_index(drop=True)
otherListings = otherListings[~otherListings["ACT Symbol"].str.startswith("File Creation Time")].reset_index(drop=True)

# Display source counts and schemas
print(f"Nasdaq listings: {len(nasdaqListings):,}")
print(f"Other US listings: {len(otherListings):,}")
print()
print("Nasdaq columns:")
print(nasdaqListings.columns.tolist())
print()
print("Other-listing columns:")
print(otherListings.columns.tolist())

Nasdaq listings: 5,592
Other US listings: 7,596

Nasdaq columns:
['Symbol', 'Security Name', 'Market Category', 'Test Issue', 'Financial Status', 'Round Lot Size', 'ETF', 'NextShares']

Other-listing columns:
['ACT Symbol', 'Security Name', 'Exchange', 'CQS Symbol', 'ETF', 'Round Lot Size', 'Test Issue', 'NASDAQ Symbol']


## Standardize Listings

In [5]:
# Define exchange names
exchangeMap = {
    "A": "NYSE American",
    "M": "NYSE Chicago",
    "N": "NYSE",
    "P": "NYSE Arca",
    "V": "IEX",
    "Z": "Cboe BZX"
}

# Validate exchange codes
unknownExchangeCodes = set(otherListings["Exchange"]) - set(exchangeMap)

if unknownExchangeCodes:
    raise ValueError(f"Unknown exchange codes: {sorted(unknownExchangeCodes)}")

# Standardize Nasdaq listings
nasdaqStandardized = nasdaqListings.rename(columns={
    "Symbol": "symbol",
    "Security Name": "securityName",
    "ETF": "isETF",
    "Test Issue": "testIssue",
    "Financial Status": "financialStatus"
})

nasdaqStandardized["exchange"] = "Nasdaq"

nasdaqStandardized = nasdaqStandardized[[
    "symbol",
    "securityName",
    "exchange",
    "isETF",
    "testIssue",
    "financialStatus"
]].copy()

# Standardize other US listings
otherStandardized = otherListings.rename(columns={
    "NASDAQ Symbol": "symbol",
    "Security Name": "securityName",
    "ETF": "isETF",
    "Test Issue": "testIssue"
})

otherStandardized["exchange"] = otherListings["Exchange"].map(exchangeMap)
otherStandardized["financialStatus"] = pd.NA

otherStandardized = otherStandardized[[
    "symbol",
    "securityName",
    "exchange",
    "isETF",
    "testIssue",
    "financialStatus"
]].copy()

In [6]:
# Combine official listing sources
usListings = pd.concat(
    [nasdaqStandardized, otherStandardized],
    ignore_index=True)

listingCountBeforeDeduplication = len(usListings)
usListings = usListings.drop_duplicates(subset=["symbol", "exchange"]).reset_index(drop=True)
duplicateListingsRemoved = listingCountBeforeDeduplication - len(usListings)

# Display combined listing counts
print(f"Combined listings: {len(usListings):,}")
print(f"Duplicate listings removed: {duplicateListingsRemoved:,}")
print()
print("Listings by exchange:")
print(usListings["exchange"].value_counts().sort_index())

Combined listings: 13,188
Duplicate listings removed: 0

Listings by exchange:
exchange
Cboe BZX         1616
IEX                 3
NYSE             2935
NYSE American     312
NYSE Arca        2729
NYSE Chicago        1
Nasdaq           5592
Name: count, dtype: int64


## Filter Listing Flags

In [7]:
# Validate listing flags
validFlagValues = {"Y", "N"}

unexpectedETFValues = set(usListings["isETF"]) - validFlagValues
unexpectedTestValues = set(usListings["testIssue"]) - validFlagValues

if unexpectedETFValues:
    raise ValueError(f"Unexpected ETF values: {sorted(unexpectedETFValues)}")

if unexpectedTestValues:
    raise ValueError(f"Unexpected test-issue values: {sorted(unexpectedTestValues)}")

# Remove ETFs and test issues
listingCountBeforeFlags = len(usListings)

stockCandidates = usListings[
    (usListings["isETF"] == "N") &
    (usListings["testIssue"] == "N")
].copy()

stockCandidates = stockCandidates.reset_index(drop=True)
listingsRemovedByFlags = listingCountBeforeFlags - len(stockCandidates)

# Display filtered counts
print(f"Listings before flag filters: {listingCountBeforeFlags:,}")
print(f"Listings after flag filters: {len(stockCandidates):,}")
print(f"Listings removed: {listingsRemovedByFlags:,}")
print()
print("Listings by exchange:")
print(stockCandidates["exchange"].value_counts().sort_index())

Listings before flag filters: 13,188
Listings after flag filters: 7,499
Listings removed: 5,689

Listings by exchange:
exchange
Cboe BZX            4
NYSE             2845
NYSE American     307
NYSE Arca          17
Nasdaq           4326
Name: count, dtype: int64


## Classify Stock Securities

In [8]:
# Define explicit equity descriptions
equityPattern = (
    r"\bCommon Stock\b|"
    r"\bCommon Shares?\b|"
    r"\bOrdinary Stock\b|"
    r"\bOrdinary Shares?\b|"
    r"\bOrdinary Share\b|"
    r"\bAmerican Depositary Shares?\b|"
    r"\bAmerican Depository Shares?\b|"
    r"\bAmerican Depositary Receipts?\b|"
    r"\bADR\b|"
    r"\bADS\b"
)

# Define explicit non-equity descriptions
nonEquityPattern = (
    r"\bWarrants?\b|"
    r"\bRights?\b|"
    r"\bUnits?\b|"
    r"\bPreferred\b|"
    r"\bPreference\b|"
    r"\bSenior Notes?\b|"
    r"\bSubordinated Notes?\b|"
    r"\bNotes? Due\b|"
    r"\bDebentures?\b|"
    r"\bETN\b"
)

securityNames = stockCandidates["securityName"]

explicitEquityMask = securityNames.str.contains(
    equityPattern,
    case=False,
    regex=True,
    na=False
)

explicitNonEquityMask = securityNames.str.contains(
    nonEquityPattern,
    case=False,
    regex=True,
    na=False
)

# Give explicit non-equity wording priority
explicitEquityMask = explicitEquityMask & ~explicitNonEquityMask

unclassifiedMask = ~explicitEquityMask & ~explicitNonEquityMask

explicitEquity = stockCandidates[explicitEquityMask].copy()
explicitNonEquity = stockCandidates[explicitNonEquityMask].copy()
unclassified = stockCandidates[unclassifiedMask].copy()

# Display classification counts
print(f"Explicit equity: {len(explicitEquity):,}")
print(f"Explicit non-equity: {len(explicitNonEquity):,}")
print(f"Unclassified: {len(unclassified):,}")
print()
print("Unclassified by exchange:")
print(unclassified["exchange"].value_counts().sort_index())
print()
print("Sample unclassified securities:")
display(
    unclassified[["symbol", "securityName", "exchange"]]
    .sort_values(["exchange", "symbol"])
    .head(30)
)

Explicit equity: 5,670
Explicit non-equity: 1,577
Unclassified: 252

Unclassified by exchange:
exchange
NYSE             181
NYSE American     10
NYSE Arca          4
Nasdaq            57
Name: count, dtype: int64

Sample unclassified securities:


,symbol,securityName,exchange
4333,AAP,Advance Auto Parts Inc.,NYSE
4352,ACEL,"Accel Entertainment, Inc.",NYSE
4376,ADX,Adams Diversified Equity Fund Inc.,NYSE
4380,AEG,Aegon Ltd. New York Registry Shares,NYSE
4389,AFB,AllianceBernstein National Municipal Income Fu...,NYSE
4475,AME,"AMETEK, Inc.",NYSE
4480,AMN,AMN Healthcare Services Inc,NYSE
4526,ARCO,Arcos Dorados Holdings Inc. Class A Shares,NYSE
4532,ARI,"Apollo Commercial Real Estate Finance, Inc",NYSE
4539,ARR,"ARMOUR Residential REIT, Inc.",NYSE


In [9]:
# Define non-stock security types
securityNames = stockCandidates["securityName"]

spacMask = securityNames.str.contains(
    r"\bAcquisition\b|\bSPAC\b",
    case=False,
    regex=True,
    na=False
)

# Identify preferred-series symbols from ACT symbology
preferredSeriesSymbols = set(
    otherListings.loc[
        otherListings["ACT Symbol"].str.contains("$", regex=False),
        "NASDAQ Symbol"
    ]
)

preferredMask = securityNames.str.contains(
    r"\bPreferred Stock\b|"
    r"\bPreferred Shares?\b|"
    r"\bPreferred Securities\b|"
    r"\bPreference Shares?\b|"
    r"\bPfd\b|"
    r"\bDepositary Shares?\b.*\bPreferred\b|"
    r"\d+(?:\.\d+)?%.*\bSeries [A-Z0-9-]+\b|"
    r"\bSeries [A-Z0-9-]+\b.*\bPreferred\b",
    case=False,
    regex=True,
    na=False
    ) | stockCandidates["symbol"].isin(preferredSeriesSymbols)

unitMask = securityNames.str.contains(
    r"\bUnits?\b",
    case=False,
    regex=True,
    na=False
)

warrantMask = securityNames.str.contains(
    r"\bWarrants?\b",
    case=False,
    regex=True,
    na=False
)

rightMask = securityNames.str.contains(
    r"(?:-\s*Rights?\b|\bRights?,\s*each\b|\bRights?$)",
    case=False,
    regex=True,
    na=False
) | stockCandidates["symbol"].str.endswith("^")

debtMask = securityNames.str.contains(
    r"\bSenior Notes?\b|"
    r"\bSubordinated Notes?\b|"
    r"\bNotes? Due\b|"
    r"\bDebentures?\b|"
    r"\bBonds? Due\b|"
    r"\bExchange-Traded Notes?\b|"
    r"\bETNs?\b|"
    r"\bTrust Certificates?\b|"
    r"\bZONES\b",
    case=False,
    regex=True,
    na=False
)

fundBdcMask = securityNames.str.contains(
    r"\bFund\b|"
    r"\bClosed[- ]End\b|"
    r"\bBDC\b|"
    r"\bBusiness Development (?:Company|Corporation)\b",
    case=False,
    regex=True,
    na=False
)

partnershipMask = securityNames.str.contains(
    r"\bLimited Partner Interests?\b|"
    r"\bLimited Partnership Interests?\b|"
    r"\bLimited Liability Company Interests?\b",
    case=False,
    regex=True,
    na=False
)

# Preserve REIT equity while removing obvious non-equity trusts
reitMask = securityNames.str.contains(
    r"\bREIT\b|"
    r"\bReal Estate Investment Trust\b|"
    r"\bRealty Trust\b|"
    r"\bHospitality Trust\b|"
    r"\bResidential Trust\b",
    case=False,
    regex=True,
    na=False
)

nonEquityTrustMask = securityNames.str.contains(
    r"\bRoyalty Trust\b|"
    r"\bGold Trust\b|"
    r"\bSilver Trust\b|"
    r"\bCommodity Trust\b|"
    r"\bBond Trust\b|"
    r"\bIncome Trust\b|"
    r"\bMunicipal.*Trust\b|"
    r"\bCredit.*Trust\b|"
    r"\bDividend Trust\b|"
    r"\bAllocation.*Trust\b|"
    r"\bTarget Term Trust\b|"
    r"\bTerm Trust\b|"
    r"\bOpportunities.*Trust\b|"
    r"\bResources Trust\b|"
    r"\bUtility.*Trust\b|"
    r"\bHealth Sciences Trust\b|"
    r"\bLife Sciences Trust\b",
    case=False,
    regex=True,
    na=False
) & ~reitMask

beneficialInterestMask = securityNames.str.contains(
    r"\bShares of Beneficial Interest\b",
    case=False,
    regex=True,
    na=False
) & ~reitMask

In [10]:
# Apply mutually exclusive exclusion reasons
classifiedListings = stockCandidates.copy()
classifiedListings["exclusionReason"] = pd.NA

exclusionRules = [
    ("SPAC", spacMask),
    ("Preferred", preferredMask),
    ("Unit", unitMask),
    ("Warrant", warrantMask),
    ("Right", rightMask),
    ("Debt", debtMask),
    ("Fund or BDC", fundBdcMask),
    ("Partnership or LLC interest", partnershipMask),
    ("Non-equity trust", nonEquityTrustMask | beneficialInterestMask)
]

for reason, mask in exclusionRules:
    newExclusions = mask & classifiedListings["exclusionReason"].isna()
    classifiedListings.loc[newExclusions, "exclusionReason"] = reason

stockUniverse = classifiedListings[classifiedListings["exclusionReason"].isna()].copy()

# Display classification summary
classificationSummary = classifiedListings["exclusionReason"].fillna("Stock").value_counts()

print(classificationSummary.to_string())
print(f"\nStock universe candidates: {len(stockUniverse):,}")

exclusionReason
Stock                          5289
SPAC                            733
Preferred                       509
Fund or BDC                     295
Warrant                         291
Debt                            172
Unit                            105
Non-equity trust                 79
Right                            21
Partnership or LLC interest       5

Stock universe candidates: 5,289


## Format Symbols

In [11]:
# Convert exchange symbols to Yahoo formatting
stockUniverse["yahooSymbol"] = stockUniverse["symbol"].str.replace(".", "-", regex=False)

## Validate and Save Universe

In [12]:
# Build final symbol list
pairDivergenceUniverse = stockUniverse["yahooSymbol"].str.strip().copy()

# Validate final universe
missingSymbolCount = pairDivergenceUniverse.isna().sum() + (pairDivergenceUniverse == "").sum()
duplicateSymbolCount = pairDivergenceUniverse.duplicated().sum()

if missingSymbolCount:
    raise ValueError(f"Missing symbols: {missingSymbolCount:,}")

if duplicateSymbolCount:
    raise ValueError(f"Duplicate symbols: {duplicateSymbolCount:,}")

pairDivergenceUniverse = pairDivergenceUniverse.sort_values().reset_index(drop=True)

# Save one symbol per line
outputPath = universeDirectory / "pairDivergenceUniverse.txt"
pairDivergenceUniverse.to_csv(outputPath, index=False, header=False)

print(f"Final universe: {len(pairDivergenceUniverse):,}")
print(f"Missing symbols: {missingSymbolCount:,}")
print(f"Duplicate symbols: {duplicateSymbolCount:,}")
print(f"Saved to: {outputPath}")

Final universe: 5,289
Missing symbols: 0
Duplicate symbols: 0
Saved to: /Users/suhas/Documents/github/Quant-Research-Lab/data/universes/pairDivergenceUniverse.txt
